In [ ]:
from huggingface_hub import login
login()

In [ ]:
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

In [ ]:
base_prompt = "Explain unlearning in one sentence."

prompt_variants = [
    "Explain unlearning in one sentence.",
    "Explain unlearning in one sentnce.",
    "Explain unlearning in 1 sentence.",
    "EXPLAIN unlearning in one sentence.",
    "Explain unlearning in one sentence!!",
]

In [ ]:
tempeeratures = [0.2, 0.7, 1.2]
runs_per_condition = 5

records = []

In [ ]:
for prompt_id, prompt in enumerate(prompt_variants):

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    input_tokens = inputs["input_ids"].shape[-1]

    for temp in temperatures:

        for run_id in range(runs_per_condition):

            if torch.cuda.is_available():

                torch.cuda.synchronize()

            start = time.perf_counter()

            output_ids = model.generate(

                **inputs,

                max_new_tokens=80,

                do_sample=True,

                temperature=temp,

                top_p=0.95,

                pad_token_id=tokenizer.eos_token_id,

            )

            if torch.cuda.is_available():

                torch.cuda.synchronize()

            elapsed = time.perf_counter() - start

            total_tokens = output_ids.shape[-1]

            generated_tokens = total_tokens - input_tokens

            tokens_per_second = generated_tokens / elapsed

            full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

            generated_text = full_text[len(prompt):].strip()

            records.append({

                "prompt_id": prompt_id,

                "prompt": prompt,

                "temperature": temp,

                "run_id": run_id,

                "input_tokens": input_tokens,

                "generated_tokens": generated_tokens,

                "elapsed_seconds": elapsed,

                "tokens_per_second": tokens_per_second,

                "generated_text": generated_text,

                "full_text": full_text,

            })

df = pd.DataFrame(records)

df.to_csv("generations.csv", index=False)

df.head()

In [ ]:
summary = (
    df.groupby("temperature")
    .agg(
        avg_tokens_per_second=("tokens_per_second", "mean"),
        std_tokens_per_second=("tokens_per_second", "std"),
        avg_generated_tokens=("generated_tokens", "mean"),
    )
    .reset_index()
)

summary.to_csv("summary_metrics.csv", index=False)
summary